### Middleware

Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

    Tracking agent behavior with logging, analytics, and debugging.
    Transforming prompts, tool selection, and output formatting.
    Adding retries, fallbacks, and early termination logic.
    Applying rate limits, guardrails, and PII detection.


In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.6-27b")    

Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

    Long-running conversations that exceed context windows.
    Multi-turn dialogues with extensive history.
    Applications where preserving full conversation context matters.


In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)


In [4]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [5]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='a0333bdc-6787-4477-b575-cb7186384b62'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Question:** This is a basic arithmetic question.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Accuracy:** The calculation is correct. No tricks or hidden meanings.\n6.  **Output Generation:** "2 + 2 equals 4." (or simply "4")\n\nI\'ll keep it direct and accurate.✅\n</think>\n\n2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 17, 'total_tokens': 161, 'completion_time': 0.274002238, 'completion_tokens_details': None, 'prompt_time': 0.000931719, 'prompt_tokens_details': None, 'queue_time': 0.04813442, 'total_time': 0.274933957}, 'model_name': 'qwen/qwen3

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token


In [7]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~151 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='87c73d8d-4f00-4102-8075-a4f2dfaef808'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to find hotels in Paris.\n2.  **Identify Available Tools**: I have a `search_hotels` tool that takes a `city` parameter.\n3.  **Match Intent to Tool**: The tool matches the intent perfectly.\n4.  **Extract Parameters**: `city` = "Paris".\n5.  **Execute Tool Call**: Call `search_hotels(city="Paris")`.\n6.  **Format Response**: Prepare the function call. Ensure correct syntax. No extra text. Wait for tool output. (Self-correction/Refinement: I will just generate the tool call now).✅\n', 'tool_calls': [{'id': 'wnmsnsj0p', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 174, 'prompt_tokens': 277, 'tota